# Лабораторная работа 11. PySpark

**Задание:**
1. Классификация цветков ирисов с использованием PySpark
2. Классификация пассажиров титаника с использованием PySpark

только pyspark.ml

## Инициализация PySpark

In [ ]:
import os

JAVA_HOME = r"C:\Program Files\Java\jdk-17"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: C:\Program Files\Java\jdk-17


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

---
# Часть 1. Классификация цветков ирисов

**Датасет Iris** содержит 150 записей о трёх видах ирисов (Setosa, Versicolor, Virginica).  
Каждый цветок описан 4 числовыми признаками:

`sepal_length` / `sepal_width` — длина и ширина чашелистика <br/>
`petal_length` / `petal_width` — длина и ширина лепестка

**Задача:** по 4 признакам предсказать вид цветка (3 класса).

### Загрузка данных

In [ ]:
iris_df = spark.read.csv('iris.csv', inferSchema=True, header=True)
iris_df = iris_df.toDF('sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'variety')

iris_df.show(5)
iris_df.printSchema()
print('Количество записей по каждому классу:')
iris_df.groupBy('variety').count().show()

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|variety|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3.1|         1.5|        0.2| Setosa|
|         5.0|        3.6|         1.4|        0.2| Setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- variety: string (nullable = true)

Количество записей по каждому классу:
+----------+-----+
|   variety|count|
+----------+-----+
| Virginica|   50|
|    Setosa|   50|
|Versicolor|   50|
+----------+-----+



### Предобработка и обучение модели

PySpark ML требует два обязательных столбца:
- **`label`** — числовой номер класса (не строка)
- **`features`** — все признаки в виде одного вектора

Для этого используем три инструмента:

| Инструмент | Что делает |
|---|---|
| `StringIndexer` | Переводит строку `variety` в число: Setosa→0, Versicolor→1, Virginica→2 |
| `VectorAssembler` | Собирает 4 числовых столбца в один столбец-вектор `features` |
| `LogisticRegression` | Обучает классификатор на столбцах `features` и `label`. Настройка модели |

**Pipeline** — это конвейер, который применяет все шаги последовательно: сначала индексирует метки, потом собирает вектор, потом обучает модель.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

label_indexer = StringIndexer(inputCol='variety', outputCol='label')

assembler = VectorAssembler(
    inputCols=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    outputCol='features'
)

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)

iris_pipeline = Pipeline(stages=[label_indexer, assembler, lr])

iris_train, iris_test = iris_df.randomSplit([0.8, 0.2], seed=42)
iris_model = iris_pipeline.fit(iris_train)
print('Модель обучена')

Модель обучена


### Предсказания и оценка качества

In [ ]:
iris_pred = iris_model.transform(iris_test)
iris_pred.select('variety', 'label', 'prediction').show()

+----------+-----+----------+
|   variety|label|prediction|
+----------+-----+----------+
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|Versicolor|  0.0|       0.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|Versicolor|  0.0|       0.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|    Setosa|  2.0|       2.0|
|Versicolor|  0.0|       0.0|
|Versicolor|  0.0|       0.0|
|Versicolor|  0.0|       0.0|
| Virginica|  1.0|       1.0|
|Versicolor|  0.0|       0.0|
| Virginica|  1.0|       1.0|
| Virginica|  1.0|       1.0|
+----------+-----+----------+
only showing top 20 rows


In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
accuracy = evaluator.evaluate(iris_pred)

total   = iris_pred.count()
correct = iris_pred.filter(F.col('label') == F.col('prediction')).count()

print(f'Iris — Accuracy: {accuracy * 100:.1f}%')
print(f'Правильно: {correct} из {total}')
print('\nМатрица ошибок (строка = реальный класс, столбец prediction = предсказанный):')
iris_pred.groupBy('variety', 'prediction').count().orderBy('variety').show()

Iris — Accuracy: 100.0%
Правильно: 24 из 24

Матрица ошибок (строка = реальный класс, столбец prediction = предсказанный):
+----------+----------+-----+
|   variety|prediction|count|
+----------+----------+-----+
|    Setosa|       2.0|   11|
|Versicolor|       0.0|    6|
| Virginica|       1.0|    7|
+----------+----------+-----+



---
# Часть 2. Классификация пассажиров Титаника

**Датасет Titanic** содержит данные о 891 пассажире.  
Каждый пассажир описан несколькими признаками:

| Признак | Описание |
|---|---|
| `Survived` | **Метка класса**: 1 = выжил, 0 = погиб |
| `Pclass` | Класс билета (1, 2, 3) |
| `Sex` | Пол (male / female) |
| `Age` | Возраст |
| `SibSp` | Кол-во братьев/сестёр и супругов на борту |
| `Parch` | Кол-во родителей и детей на борту |
| `Fare` | Стоимость билета |
| `Embarked` | Порт посадки (S / C / Q) |

**Задача:** предсказать выжил пассажир или нет (2 класса — бинарная классификация).

### Загрузка данных

In [ ]:
titanic_df = spark.read.csv('titanic.csv', inferSchema=True, header=True)
titanic_df.show(5)
titanic_df.printSchema()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

### Предобработка данных

В данных есть пропуски (особенно в `Age` и `Embarked`) — удаляем такие строки через `dropna()`  
Оставляем только те признаки, которые имеют смысл для предсказания выживания.

In [19]:
titanic_clean = titanic_df.select('Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked').dropna()

print('Строк после удаления пропусков:', titanic_clean.count())
print('Распределение по классу (0=погиб, 1=выжил):')
titanic_clean.groupBy('Survived').count().show()

Строк после удаления пропусков: 712
Распределение по классу (0=погиб, 1=выжил):
+--------+-----+
|Survived|count|
+--------+-----+
|       1|  288|
|       0|  424|
+--------+-----+



### Построение Pipeline

В Titanic есть строковые категориальные признаки (`Sex`, `Embarked`), которые нужно закодировать:

| Шаг | Инструмент | Что делает |
|---|---|---|
| 1 | `StringIndexer` | `Sex`: male→0, female→1 / `Embarked`: S→0, C→1, Q→2 |
| 2 | `OneHotEncoder` | Превращает индексы в бинарные векторы — убирает ложную «упорядоченность» чисел |
| 3 | `VectorAssembler` | Собирает все признаки в один вектор `features` |
| 4 | `LogisticRegression` | Обучает бинарный классификатор |

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

sex_indexer      = StringIndexer(inputCol='Sex',      outputCol='sex_idx')
embarked_indexer = StringIndexer(inputCol='Embarked', outputCol='embarked_idx')

sex_encoder      = OneHotEncoder(inputCol='sex_idx',      outputCol='sex_ohe')
embarked_encoder = OneHotEncoder(inputCol='embarked_idx', outputCol='embarked_ohe')

assembler = VectorAssembler(
    inputCols=['Pclass', 'sex_ohe', 'Age', 'SibSp', 'Parch', 'Fare', 'embarked_ohe'],
    outputCol='features'
)

lr = LogisticRegression(featuresCol='features', labelCol='Survived', maxIter=100)

titanic_pipeline = Pipeline(stages=[
    sex_indexer, embarked_indexer,
    sex_encoder, embarked_encoder,
    assembler, lr
])

titanic_train, titanic_test = titanic_clean.randomSplit([0.8, 0.2], seed=42)
titanic_model = titanic_pipeline.fit(titanic_train)
print('Модель обучена')

Модель обучена


### Предсказания и оценка качества

In [ ]:
titanic_pred = titanic_model.transform(titanic_test)
titanic_pred.select('Survived', 'prediction', 'probability').show(10)

+--------+----------+--------------------+
|Survived|prediction|         probability|
+--------+----------+--------------------+
|       0|       1.0|[0.07737986434598...|
|       0|       1.0|[0.35668705657654...|
|       0|       1.0|[0.26231992481583...|
|       0|       1.0|[0.44283163781661...|
|       0|       1.0|[0.37425696446519...|
|       0|       0.0|[0.54008635603974...|
|       0|       0.0|[0.83655771934375...|
|       0|       0.0|[0.71037916225538...|
|       0|       0.0|[0.68846865655073...|
|       0|       0.0|[0.68504416736650...|
+--------+----------+--------------------+
only showing top 10 rows


In [ ]:
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='Survived', predictionCol='prediction', metricName='accuracy'
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol='Survived', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

accuracy = acc_evaluator.evaluate(titanic_pred)
auc      = auc_evaluator.evaluate(titanic_pred)

total   = titanic_pred.count()
correct = titanic_pred.filter(
    F.col('Survived') == F.col('prediction').cast('int')
).count()

print(f'Titanic — Accuracy: {accuracy * 100:.1f}%')
print(f'Titanic — AUC-ROC:  {auc:.4f}')
print(f'Правильно: {correct} из {total}')
print('\nМатрица ошибок:')
titanic_pred.groupBy('Survived', 'prediction').count().orderBy('Survived').show()

Titanic — Accuracy: 78.9%
Titanic — AUC-ROC:  0.8679
Правильно: 90 из 114

Матрица ошибок:
+--------+----------+-----+
|Survived|prediction|count|
+--------+----------+-----+
|       0|       0.0|   49|
|       0|       1.0|   11|
|       1|       0.0|   13|
|       1|       1.0|   41|
+--------+----------+-----+

